In [ ]:
# ============================================================
# Figure 4 momentum-distribution calculation
# ============================================================
#
# This notebook computes the impurity momentum distributions for
#
#   beta / beta_c1 = 1 and 0.2
#
# at the times used in Fig. 4.
#
# The momentum points are uniformly spaced in log(k):
#
#   k_j = k_min * (k_max / k_min) ** [j / (N_k - 1)].
#
# The oscillatory time integral is evaluated exactly for the
# piecewise-linear interpolant of chi_q(t). An oversampled fast Fourier
# transform is used to evaluate the transform efficiently, and the
# angular integral is converted into an integral over energy.

from pathlib import Path

import numpy as np
from tqdm.auto import tqdm

from fermi_polaron_trapezoid_optimized import (
    SolverConfig,
    run_simulation,
    validate_result,
)


# ============================================================
# User settings
# ============================================================

OUTPUT_DIR = Path.cwd()

BETA_C1 = 2.0 * np.sqrt(2.0 / np.pi)
T_F = 2.0

# Grid used to solve Eq. (6) for chi_q(t).
N_Q = 100
DT = 1.0 / 200.0
N_SEED = 14

# Log-uniform momentum grid.
K_MIN = 2.0
K_MAX = 100.0
N_K_POINTS = 40

# FFT zero-padding factor. Compare 8, 16, and 32 for convergence.
FFT_OVERSAMPLING = 16

# Ignore an existing chi cache and solve Eq. (6) again.
RECOMPUTE_CHI = False

# Recompute n_k even when a matching output file already exists.
RECOMPUTE_NK = True

SHOW_PROGRESS = True

# beta/beta_c1, times in units of t_F, output label.
CASES = [
    (1.0, (5.0, 50.0), "beta_1"),
    (0.2, (5.0, 30.0, 50.0), "beta_0p2"),
]

MAX_TIME = max(max(times) for _, times, _ in CASES) * T_F
N_T = int(round(MAX_TIME / DT))
K_GRID = np.geomspace(K_MIN, K_MAX, N_K_POINTS)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Figure 4 numerical settings")
print(f"  output directory   = {OUTPUT_DIR.resolve()}")
print(f"  dt                 = {DT:g}")
print(f"  n_q                = {N_Q}")
print(f"  n_t                = {N_T}")
print(f"  FFT oversampling   = {FFT_OVERSAMPLING}")
print(f"  number of k points = {N_K_POINTS}")
print(f"  k range            = [{K_GRID[0]:g}, {K_GRID[-1]:g}]")
print(f"  adjacent k ratio   = {K_GRID[1] / K_GRID[0]:.10f}")

In [ ]:
# ============================================================
# Oscillatory Fourier integral
# ============================================================
#
# For fixed q, define
#
#   I_q(E, T) = integral_0^T d tau chi_q(tau) exp(i E tau).
#
# The following functions integrate the oscillatory exponential exactly
# for the piecewise-linear interpolant of chi_q(t), evaluate the transform
# on an oversampled FFT energy grid, and construct
#
#   F_q(E) = integral_0^E dE' |I_q(E')|^2.

def next_power_of_two(n: int) -> int:
    """Return the smallest power of two greater than or equal to n."""

    return 1 << (int(n) - 1).bit_length()

def endpoint_basis_b(energy: np.ndarray, dt: float) -> np.ndarray:
    r"""Fourier transform of the right endpoint linear basis function.

    B(E) = int_0^dt ds (s/dt) exp(i E s).

    A short series is used near E*dt=0 to avoid catastrophic cancellation.
    """

    x = np.asarray(energy, dtype=float) * dt
    result = np.empty(x.shape, dtype=np.complex128)
    small = np.abs(x) < 1.0e-4

    xs = x[small]
    result[small] = dt * (
        0.5
        + 1.0j * xs / 3.0
        - xs**2 / 8.0
        - 1.0j * xs**3 / 30.0
        + xs**4 / 144.0
    )

    xl = x[~small]
    result[~small] = (
        dt * (np.exp(1.0j * xl) * (1.0 - 1.0j * xl) - 1.0) / xl**2
    )
    return result

def evaluate_piecewise_linear_transform_on_fft_grid(
    spectrum: np.ndarray,
    endpoint_value: complex,
    energy_indices: np.ndarray,
    delta_energy: float,
    n_intervals: int,
    dt: float,
) -> np.ndarray:
    r"""Evaluate I(E)=int_0^T chi(t) exp(iEt)dt on the FFT frequency grid.

    ``spectrum[m]`` is the positive-sign discrete Fourier sum of the interior
    time samples at E_m=m*delta_energy, periodically continued with period
    2*pi/dt.  The formula below is exact for the piecewise-linear interpolant
    of chi(t) at these frequency-grid points.
    """

    indices = np.asarray(energy_indices, dtype=np.int64)
    energy = indices.astype(float) * delta_energy
    periodic_indices = np.remainder(indices, spectrum.size)
    interior_sum = spectrum[periodic_indices]

    x = energy * dt
    interior_basis = dt * np.sinc(x / (2.0 * np.pi)) ** 2
    endpoint_term = (
        endpoint_value
        * np.exp(1.0j * energy * (n_intervals - 1) * dt)
        * endpoint_basis_b(energy, dt)
    )
    return interior_basis * interior_sum + endpoint_term

def cumulative_power_spectrum(
    chi_samples: np.ndarray,
    dt: float,
    energy_max: float,
    fft_oversampling: int,
    chunk_size: int = 250_000,
) -> tuple[np.ndarray, float]:
    r"""Return cumulative integral F(E)=int_0^E |I(E')|^2 dE'.

    ``chi_samples`` contains chi(dt), ..., chi(T).  chi(0)=0 is inserted
    explicitly.  Adjacent samples are connected by straight lines.
    """

    n_intervals = chi_samples.size
    if n_intervals < 2:
        raise ValueError("At least two positive-time chi samples are required.")
    if fft_oversampling < 2:
        raise ValueError("fft_oversampling must be at least 2.")

    n_fft = next_power_of_two(fft_oversampling * (n_intervals + 1))
    delta_energy = 2.0 * np.pi / (n_fft * dt)
    n_energy = int(np.ceil(energy_max / delta_energy)) + 2

    # Interior-node trigonometric sum:
    # S(E)=sum_{j=1}^{N-1} chi_j exp(i E j dt).
    work = np.zeros(n_fft, dtype=np.complex128)
    work[1:n_intervals] = chi_samples[:-1]
    spectrum = np.fft.ifft(work) * n_fft
    endpoint_value = complex(chi_samples[-1])

    cumulative = np.empty(n_energy, dtype=float)
    cumulative[0] = 0.0

    # Compute |I(E)|^2 in chunks and accumulate its trapezoidal integral.
    previous_power: float | None = None
    previous_cumulative = 0.0
    start = 0
    while start < n_energy:
        stop = min(start + chunk_size, n_energy)
        indices = np.arange(start, stop, dtype=np.int64)
        transform = evaluate_piecewise_linear_transform_on_fft_grid(
            spectrum=spectrum,
            endpoint_value=endpoint_value,
            energy_indices=indices,
            delta_energy=delta_energy,
            n_intervals=n_intervals,
            dt=dt,
        )
        power = np.abs(transform) ** 2

        if start == 0:
            if stop > 1:
                increments = 0.5 * delta_energy * (power[:-1] + power[1:])
                cumulative[1:stop] = np.cumsum(increments)
            previous_cumulative = cumulative[stop - 1]
        else:
            bridge = 0.5 * delta_energy * (previous_power + power[0])
            cumulative[start] = previous_cumulative + bridge
            if stop > start + 1:
                increments = 0.5 * delta_energy * (power[:-1] + power[1:])
                cumulative[start + 1 : stop] = cumulative[start] + np.cumsum(
                    increments
                )
            previous_cumulative = cumulative[stop - 1]

        previous_power = float(power[-1])
        start = stop

    return cumulative, delta_energy

def interpolate_cumulative(
    cumulative: np.ndarray, delta_energy: float, energy: np.ndarray
) -> np.ndarray:
    """Linearly interpolate a cumulative integral sampled on E_j=j*dE."""

    x = np.asarray(energy, dtype=float) / delta_energy
    left = np.floor(x).astype(np.int64)
    fraction = x - left

    if np.any(left < 0) or np.any(left + 1 >= cumulative.size):
        raise ValueError("Requested energy lies outside the cumulative grid.")

    return cumulative[left] * (1.0 - fraction) + cumulative[left + 1] * fraction

In [ ]:
# ============================================================
# Momentum distribution
# ============================================================
#
# For k >= 2 k_F,
#
#   integral_{-1}^{1} d mu |I_q(k^2 - k q mu)|^2
#
#       = [F_q(k^2 + k q) - F_q(k^2 - k q)] / (k q).
#
# The remaining q integral uses the same trapezoidal momentum grid as
# the solver for chi_q(t).

def compute_nk_at_time(
    chi_qt: np.ndarray,
    q_grid: np.ndarray,
    dt: float,
    target_time: float,
    k_grid: np.ndarray,
    fft_oversampling: int = 8,
    show_progress: bool = True,
) -> np.ndarray:
    r"""Compute nbar_k=n_{k,down}/n_down at one target time.

    The q integral uses the same trapezoidal q grid as the chi solver.
    """

    target_steps = int(round(target_time / dt))
    if not np.isclose(target_steps * dt, target_time, rtol=0.0, atol=1.0e-12):
        raise ValueError("target_time must lie on the solver time grid.")
    if target_steps < 2 or target_steps > chi_qt.shape[1]:
        raise ValueError("target_time is outside the available chi_qt history.")

    k_grid = np.asarray(k_grid, dtype=float)
    q_grid = np.asarray(q_grid, dtype=float)
    if np.any(k_grid < 2.0 * q_grid[-1] - 1.0e-12):
        raise ValueError("This implementation assumes k >= 2*k_F.")

    dq = q_grid[1] - q_grid[0]
    energy_max = float(np.max(k_grid**2 + k_grid * q_grid[-1]))
    q_integral = np.zeros(k_grid.shape, dtype=float)

    iterator = range(1, q_grid.size)  # q=0 has zero radial measure
    if show_progress:
        iterator = tqdm(
            iterator,
            desc=rf"Computing n_k at t/t_F={target_time / T_F:g}",
            leave=False,
        )

    for iq in iterator:
        q = q_grid[iq]
        cumulative, delta_energy = cumulative_power_spectrum(
            chi_samples=chi_qt[iq, :target_steps],
            dt=dt,
            energy_max=energy_max,
            fft_oversampling=fft_oversampling,
        )

        lower = k_grid**2 - k_grid * q
        upper = k_grid**2 + k_grid * q
        interval_integral = interpolate_cumulative(
            cumulative, delta_energy, upper
        ) - interpolate_cumulative(cumulative, delta_energy, lower)

        angular_integral = interval_integral / (k_grid * q)
        q_endpoint_weight = 0.5 if iq == q_grid.size - 1 else 1.0
        q_integral += q_endpoint_weight * q**2 * angular_integral

    # nbar_k = 1/(4*pi^2) int dq q^2 int dmu |I|^2.
    n_k = dq * q_integral / (4.0 * np.pi**2)

    # Only tiny negative values caused by roundoff may be clipped.
    scale = max(1.0, float(np.max(np.abs(n_k))))
    negative_tolerance = 1.0e-12 * scale
    minimum_nk = float(np.min(n_k))

    if minimum_nk < -negative_tolerance:
        raise FloatingPointError(
            f"n_k contains a significant negative value: {minimum_nk:.6e}"
        )

    n_k[n_k < 0.0] = 0.0
    return n_k

In [ ]:
# ============================================================
# Cache handling
# ============================================================
#
# Solving Eq. (6) is considerably more expensive than reconstructing
# n_k. A cached chi_q(t) history is reused only when all relevant
# numerical parameters match the current settings.

def load_or_compute_chi(beta_ratio, label):
    """Load a compatible chi cache or solve Eq. (6)."""

    beta = beta_ratio * BETA_C1
    cache_file = OUTPUT_DIR / f"figure4_{label}_chi_cache.npz"

    config = SolverConfig(
        beta=beta,
        kf=1.0,
        n_q=N_Q,
        dt=DT,
        n_t=N_T,
        n_seed=N_SEED,
        show_progress=SHOW_PROGRESS,
    )

    if cache_file.exists() and not RECOMPUTE_CHI:
        with np.load(cache_file) as data:
            required = {
                "time", "q", "chi_qt", "contact",
                "beta", "kf", "dt", "n_q", "n_t", "n_seed",
            }

            parameters_match = (
                required.issubset(data.files)
                and np.isclose(float(data["beta"]), beta)
                and np.isclose(float(data["kf"]), 1.0)
                and np.isclose(float(data["dt"]), DT)
                and int(data["n_q"]) == N_Q
                and int(data["n_t"]) == N_T
                and int(data["n_seed"]) == N_SEED
            )

            if parameters_match:
                print(f"Loading chi cache: {cache_file.name}")
                return (
                    data["q"].copy(),
                    data["chi_qt"].copy(),
                    data["contact"].copy(),
                )

        print(f"Ignoring incompatible cache: {cache_file.name}")

    print(f"Solving Eq. (6) for beta/beta_c1={beta_ratio:g}")
    result = run_simulation(config)
    validate_result(result)

    np.savez_compressed(
        cache_file,
        time=result.time,
        q=result.q,
        chi_qt=result.chi_qt,
        contact=result.contact,
        beta=beta,
        kf=1.0,
        dt=DT,
        n_q=N_Q,
        n_t=N_T,
        n_seed=N_SEED,
    )

    print(f"  evolution time = {result.elapsed_seconds:.2f} s")
    print(
        "  minimum |Sherman-Morrison denominator| = "
        f"{result.min_rank1_denominator:.6e}"
    )
    print(f"  saved cache: {cache_file.name}")

    return (
        result.q.copy(),
        result.chi_qt.copy(),
        result.contact.copy(),
    )

In [ ]:
# ============================================================
# Run the two Figure 4 cases
# ============================================================
#
# Existing n_k data are reused only when the momentum grid, requested
# times, FFT oversampling factor, and main solver parameters match the
# current settings.

output_files = []

for beta_ratio, times_over_t_f, label in CASES:
    output_file = OUTPUT_DIR / f"figure4_{label}_nk.npz"
    times_array = np.asarray(times_over_t_f, dtype=float)

    use_existing_nk = False

    if output_file.exists() and not RECOMPUTE_NK:
        with np.load(output_file) as data:
            required = {
                "k", "times_over_t_f", "n_k", "contact_at_times",
                "fft_oversampling", "dt", "n_q", "n_t", "n_seed",
            }

            if required.issubset(data.files):
                same_k = (
                    data["k"].shape == K_GRID.shape
                    and np.allclose(
                        data["k"], K_GRID,
                        rtol=1.0e-13,
                        atol=1.0e-14,
                    )
                )
                same_times = (
                    data["times_over_t_f"].shape == times_array.shape
                    and np.allclose(
                        data["times_over_t_f"], times_array,
                        rtol=0.0,
                        atol=1.0e-14,
                    )
                )
                same_parameters = (
                    int(data["fft_oversampling"]) == FFT_OVERSAMPLING
                    and np.isclose(float(data["dt"]), DT)
                    and int(data["n_q"]) == N_Q
                    and int(data["n_t"]) == N_T
                    and int(data["n_seed"]) == N_SEED
                )
                use_existing_nk = same_k and same_times and same_parameters

    if use_existing_nk:
        print(f"Using existing n_k data: {output_file.name}")
        output_files.append(output_file)
        continue

    q_grid, chi_qt, contact = load_or_compute_chi(beta_ratio, label)

    n_k_curves = []
    contact_at_times = []

    for time_over_t_f in times_over_t_f:
        target_time = time_over_t_f * T_F

        n_k = compute_nk_at_time(
            chi_qt=chi_qt,
            q_grid=q_grid,
            dt=DT,
            target_time=target_time,
            k_grid=K_GRID,
            fft_oversampling=FFT_OVERSAMPLING,
            show_progress=SHOW_PROGRESS,
        )

        contact_index = int(round(target_time / DT)) - 1
        contact_value = float(contact[contact_index])

        n_k_curves.append(n_k)
        contact_at_times.append(contact_value)

        tail_ratio = n_k * K_GRID**4 / contact_value
        high_k_mask = K_GRID >= 80.0

        print(
            f"  beta/beta_c1={beta_ratio:g}, "
            f"t/t_F={time_over_t_f:g}: "
            "median[k^4 n_k/C] for k>=80 = "
            f"{np.median(tail_ratio[high_k_mask]):.6f}"
        )

    np.savez_compressed(
        output_file,
        k=K_GRID,
        times_over_t_f=times_array,
        internal_times=T_F * times_array,
        n_k=np.asarray(n_k_curves),
        contact_at_times=np.asarray(contact_at_times),
        beta=beta_ratio * BETA_C1,
        beta_ratio=beta_ratio,
        beta_c1=BETA_C1,
        kf=1.0,
        dt=DT,
        n_q=N_Q,
        n_t=N_T,
        n_seed=N_SEED,
        fft_oversampling=FFT_OVERSAMPLING,
        normalization="n_k_down / n_down",
        k_grid_type="log_uniform",
    )

    print(f"Saved n_k data: {output_file.name}")
    output_files.append(output_file)

print("\nFinished.")
for path in output_files:
    print(" ", path.resolve())